# 🧪 Research capstone: from crowded scene to evidence

**Xu Lab | Biology-to-CryoEM Research | Module 26**  
**Phase:** G: Integrative research | **Level:** scaffolded undergraduate → research | **Time:** 60–100 minutes  
**Hardware:** Google Colab free-tier CPU (optional T4 GPU for later extensions)  
**Core tools:** Python, NumPy, SciPy, Matplotlib; selected modules use scikit-learn/PyTorch  
**Prerequisites:** Previous tutorial  
**Status:** self-contained *pedagogical demonstration* using synthetic data, not an official reproduction of Xu Lab publications.  
**Safety:** for research and educational purposes only, not for clinical diagnosis.

## Learning objectives

1. Explain the biological question and physical limitations.
2. Derive or interpret the model below and identify its assumptions.
3. Run the experiment, change a parameter, and interpret quantitative output.
4. Explain how the concept supports the research papers linked below.

## Biological motivation

Combine detection, denoising, localization, and biological interpretation in one evidence-based mini-study.

## Conceptual bridge

Generate two held-out synthetic test scenes, pick candidate particles, and report object-level precision and recall at a localization tolerance. This is a teaching pipeline, not a replication benchmark.

## Mathematical model

$\mathrm{recall}=TP/N_{\mathrm{truth}}$, $\mathrm{precision}=TP/N_{\mathrm{pred}}$.

## Notebook roadmap

1. Reproducible setup  2. Synthetic experiment  3. Interpretation  4. Challenge  5. Paper connection.


In [ ]:
# -- Environment check and shared utilities --
import sys, importlib.util
import numpy as np
import scipy
from scipy import ndimage as ndi, signal
import matplotlib.pyplot as plt
SEED=2026
rng=np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi':95,'axes.grid':False})
print('Python:',sys.version.split()[0], 'NumPy:',np.__version__)
print('PyTorch available:',bool(importlib.util.find_spec('torch')))
try:
    import torch
    print('CUDA available:',torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch not installed; required only for Module 13/14')
# Colab stock environment normally contains the required libraries.
# If an import fails, run: %pip install -q numpy scipy matplotlib scikit-learn torch

def phantom(n=48):
    """Build a deterministic 3D synthetic density with three distinct macromolecular blobs."""
    zz, yy, xx = np.mgrid[:n,:n,:n].astype(float)
    rho = np.zeros((n,n,n),dtype=float)
    for z,y,x,s,amp in [(0.45*n,.47*n,.48*n,.10*n,1.),(.65*n,.38*n,.58*n,.07*n,.7),(.40*n,.66*n,.36*n,.075*n,.8)]:
        rho += amp*np.exp(-((zz-z)**2+(yy-y)**2+(xx-x)**2)/(2*s*s))
    return rho/np.max(rho)


## Experiment: build, inspect, and check a scientific claim

Run the code below, then identify (i) input, (ii) transformation, (iii) output and (iv) a key assumption.

In [ ]:
def make_test(seed):
 rr=np.random.default_rng(seed);im=rr.normal(0,.12,(96,96));yy,xx=np.mgrid[-4:5,-4:5];temp=np.exp(-(xx**2+yy**2)/8)
 positions=[(20,22),(40,63),(73,36)]
 for y,x in positions:im[y-4:y+5,x-4:x+5]+=temp
 return im,np.array(positions),temp
for seed in [SEED+100,SEED+101]:
 im,truth,tpl=make_test(seed);sc=signal.correlate2d(im,tpl,mode='same');loc=np.argwhere((sc==ndi.maximum_filter(sc,size=11))&(sc>sc.max()*.6))
 matches=0;used=set()
 for p in loc:
  d=np.linalg.norm(truth-p,axis=1);j=int(np.argmin(d))
  if d[j]<=3 and j not in used:matches+=1;used.add(j)
 precision=matches/len(loc) if len(loc) else 0;recall=matches/len(truth)
 print(f'seed={seed}: predictions={len(loc)} precision={precision:.2f} recall={recall:.2f}')
assert len(truth)==3

## Explain what happened

**Expected scientific observation:** Separate image-level and particle-level metrics; compare external tomograms, conduct ablations, and explicitly avoid claims unsupported by observed data.

Write 3–5 sentences connecting the plotted result to the biological motivation. Describe one failure case and one alternative explanation.

## Guided challenge (edit code above)

Create a one-page research report: define biological question, specimen, simulated acquisition, method, data split, metric, failure modes and which paper you would read next.

**Deliverable:** a labeled figure or small results table, a claim, and one limitation. Quantify the effect rather than just saying “it looks better.”

## Research connection and reading strategy

**Related Xu Lab research:** 2019 de novo mining; 2023 DISCA; 2025 AITom; 2026 DUIL.

Read each paper in this order: biological question → observations/data → forward model → key objective → baseline → evaluation split → limitations. Distinguish what the toy exercise illustrates from the actual published implementation. See `../PAPER_TO_TUTORIAL_MAP.md` for publication titles and further prerequisites.

## Exit ticket

1. What is known from the measured data, versus assumed by the model?
2. Which plotted quantity could be misleading if evaluated only on synthetic data?
3. Sketch one controlled ablation and one biological validation experiment.
4. Save an updated notebook and a 150-word lab note.